# NASDAQ-100 pipeline: data check & research

Load events and daily bars, inspect data, run the pipeline, and explore results.

## Setup

Run from repo root so `src` is on the path, or install the package: `uv sync && uv run jupyter notebook`.

In [14]:
import sys
from pathlib import Path

# Ensure package is importable: add repo root / src when running from notebooks/
root = Path.cwd() if (Path.cwd() / "src" / "hedging_pipeline").exists() else Path.cwd().parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

import pandas as pd
from hedging_pipeline.config import DEFAULT_EVENTS_PATH, DEFAULT_BARS_PATH, PROJECT_ROOT
from hedging_pipeline.loaders import EventLoader
from hedging_pipeline.classification import EventClassifier
from hedging_pipeline.enrichment import PriceEnricher
from hedging_pipeline.summary import SummaryStats
from hedging_pipeline.pipeline import Pipeline

## Load data

Paths default to project root (`nasdaq_events.xlsx`, `daily_bars.parquet`). Override if needed.

In [15]:
events_path = DEFAULT_EVENTS_PATH
bars_path = DEFAULT_BARS_PATH
print("Events:", events_path, "exists:", events_path.exists())
print("Bars:", bars_path, "exists:", bars_path.exists())

Events: /Users/vitaliiaioffe/hw/hedging-pipeline-hw/nasdaq_events.xlsx exists: True
Bars: /Users/vitaliiaioffe/hw/hedging-pipeline-hw/daily_bars.parquet exists: True


In [16]:
loader = EventLoader()
raw_events = loader.load_events(events_path)
print("Raw events shape:", raw_events.shape)
raw_events.head(10)

Raw events shape: (81, 6)


,ANN DATE AFTER CLOSE,EFF DATE MORNING OF,add,del,TRADE EST MM,type
0,2016-02-12,2016-02-22,CSX,KLAC,57.789001,adhoc
1,2016-03-09,2016-03-16,NTES,SNDK,13.072000,adhoc
2,2016-06-10,2016-06-20,XRAY,LMCK,4.684619,adhoc
3,2016-07-08,2016-07-18,MCHP,ENDP,8.592609,adhoc
4,2016-10-11,2016-10-19,SHPG,LLTC,6.019807,adhoc
5,2017-01-30,2017-02-07,JBHT,NXPI,2.243701,adhoc
6,2017-03-10,2017-03-20,IDXX,SBAC,1.760104,adhoc
7,2017-04-13,2017-04-24,WYNN,TRIP,2.038358,adhoc
8,2017-06-09,2017-06-19,MELI,YHOO,0.883147,adhoc
9,2017-10-13,2017-10-23,ALGN,MAT,1.603012,adhoc


In [25]:
events = loader.normalize_events(raw_events)
print(raw_events.head(10))
print(raw_events["type"].unique())
print("Normalized events shape:", events.shape)
print("Columns:", list(events.columns))
events.head(10)

  ANN DATE AFTER CLOSE EFF DATE MORNING OF   add   del  TRADE EST MM   type
0           2016-02-12          2016-02-22   CSX  KLAC     57.789001  adhoc
1           2016-03-09          2016-03-16  NTES  SNDK     13.072000  adhoc
2           2016-06-10          2016-06-20  XRAY  LMCK      4.684619  adhoc
3           2016-07-08          2016-07-18  MCHP  ENDP      8.592609  adhoc
4           2016-10-11          2016-10-19  SHPG  LLTC      6.019807  adhoc
5           2017-01-30          2017-02-07  JBHT  NXPI      2.243701  adhoc
6           2017-03-10          2017-03-20  IDXX  SBAC      1.760104  adhoc
7           2017-04-13          2017-04-24  WYNN  TRIP      2.038358  adhoc
8           2017-06-09          2017-06-19  MELI  YHOO      0.883147  adhoc
9           2017-10-13          2017-10-23  ALGN   MAT      1.603012  adhoc
['adhoc' 'ANNUAL']
Normalized events shape: (161, 6)
Columns: ['ann_date', 'eff_date', 'ticker', 'action', 'event_type', 'trade_est_mm']


,ann_date,eff_date,ticker,action,event_type,trade_est_mm
0,2016-02-12,2016-02-22,CSX,add,adhoc,57.789001
1,2016-02-12,2016-02-22,KLAC,del,adhoc,57.789001
2,2016-03-09,2016-03-16,NTES,add,adhoc,13.072000
3,2016-03-09,2016-03-16,SNDK,del,adhoc,13.072000
4,2016-06-10,2016-06-20,XRAY,add,adhoc,4.684619
5,2016-06-10,2016-06-20,LMCK,del,adhoc,4.684619
6,2016-07-08,2016-07-18,MCHP,add,adhoc,8.592609
7,2016-07-08,2016-07-18,ENDP,del,adhoc,8.592609
8,2016-10-11,2016-10-19,SHPG,add,adhoc,6.019807
9,2016-10-11,2016-10-19,LLTC,del,adhoc,6.019807


## Unique tickers, actions, event types

Counts and lists from normalized events.

In [20]:
unique_tickers = sorted(events["ticker"].unique())
print(f"Unique tickers ({len(unique_tickers)}):")
print(unique_tickers)

print("\nActions:")
display(events["action"].value_counts())

print("Event types:")
display(events["event_type"].value_counts())

Unique tickers (128):
['AAL', 'ABNB', 'AEP', 'AKAM', 'ALGN', 'ALNY', 'ALXN', 'AMD', 'ANSS', 'APP', 'ARM', 'ASML', 'ATVI', 'AZN', 'BIDU', 'BIIB', 'BKR', 'BMRN', 'CA', 'CCEP', 'CDNS', 'CDW', 'CELG', 'CERN', 'CHKP', 'CPRT', 'CRWD', 'CSGP', 'CSX', 'CTXS', 'DASH', 'DDOG', 'DISCA', 'DISH', 'DLTR', 'DOCU', 'DXCM', 'EBAY', 'ENDP', 'ENPH', 'ESRX', 'EXC', 'EXPE', 'FANG', 'FER', 'FOX', 'FSRV', 'FTNT', 'GEHC', 'GFS', 'HAS', 'HOLX', 'HON', 'HSIC', 'IDXX', 'INCY', 'INSM', 'JBHT', 'JD', 'KDP', 'KLAC', 'LBTYA', 'LCID', 'LIN', 'LLTC', 'LMCK', 'LULU', 'MAT', 'MCHP', 'MDB', 'MELI', 'MPWR', 'MRNA', 'MRVL', 'MTCH', 'MXIM', 'MYL', 'NCLH', 'NLOK', 'NTAP', 'NTES', 'NXPI', 'ODFL', 'OKTA', 'ON', 'PANW', 'PDD', 'PEP', 'PTON', 'QRTEA', 'RIVN', 'ROP', 'SBAC', 'SGEN', 'SHOP', 'SHPG', 'SIRI', 'SMCI', 'SNDK', 'SNPS', 'SPLK', 'STX', 'SWKS', 'TCOM', 'TEAM', 'TRI', 'TRIP', 'TSCO', 'TTD', 'TTWO', 'UAL', 'ULTA', 'VIAB', 'VOD', 'VRSN', 'WBA', 'WBD', 'WDAY', 'WDC', 'WLTW', 'WTW', 'WYNN', 'XEL', 'XLNX', 'XRAY', 'YHOO', 'ZM',

action
del    81
add    80
Name: count, dtype: int64

Event types:


event_type
annual    95
adhoc     66
Name: count, dtype: int64

In [26]:
bars = loader.load_daily_bars(bars_path)
print("Bars shape:", bars.shape)
print("Date range:", bars["date"].min(), "to", bars["date"].max())
print("Symbols:", bars["symbol"].unique()[:20])
bars.head()

Bars shape: (97321, 5)
Date range: 2020-01-02 00:00:00 to 2026-01-20 00:00:00
Symbols: ['HON' 'SIRI' 'ZS' 'FOX' 'ROP' 'BIDU' 'XLNX' 'NTES' 'SWKS' 'DOCU' 'FANG'
 'WMT' 'TTWO' 'ODFL' 'PANW' 'CDW' 'ILMN' 'ALNY' 'MXIM' 'DDOG']


,date,symbol,open,close,volume
0,2020-01-02,HON,147.63,150.37,3284762
1,2020-01-02,SIRI,59.87,59.87,1686600
2,2020-01-02,ZS,46.87,47.33,1443442
3,2020-01-02,FOX,33.49,33.45,1268445
4,2020-01-02,ROP,341.30,352.14,878923


## Quick pipeline run

Classify → enrich → summarize. Outputs go to `output/` by default.

In [27]:
pipeline = Pipeline()
enriched, summary_df, enriched_with_outliers = pipeline.run(
    events_path=events_path,
    bars_path=bars_path
)
print("Enriched shape:", enriched.shape)
enriched.head()

83 events could not be fully enriched (missing prices or dates)


Enriched shape: (161, 18)


,ann_date,eff_date,ticker,action,event_type,trade_est_mm,classification,entry_date,exit_date,entry_open,exit_close,qqq_entry_open,qqq_exit_close,holding_period_trading_days,stock_return,qqq_return,excess_return,first_day_return
0,2016-02-12,2016-02-22,CSX,add,adhoc,57.789001,adhoc_add,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-02-12,2016-02-22,KLAC,del,adhoc,57.789001,adhoc_del,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-03-09,2016-03-16,NTES,add,adhoc,13.072000,adhoc_add,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-03-09,2016-03-16,SNDK,del,adhoc,13.072000,adhoc_del,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-06-10,2016-06-20,XRAY,add,adhoc,4.684619,adhoc_add,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Classification summary & results

**Classification summary:** counts by classification label. **Results:** summary stats by group (mean excess return, win rate, etc.) from the pipeline.

In [28]:
# Classification summary (counts by label)
print("Classification counts:")
display(enriched["classification"].value_counts().sort_index())

# Results: summary by group (from pipeline)
print("\nResults — summary by group:")
if summary_df is not None:
    display(summary_df)
else:
    summary_path = PROJECT_ROOT / "output" / "summary_by_group.csv"
    if summary_path.exists():
        display(pd.read_csv(summary_path))
    else:
        summary = SummaryStats().compute_summary_by_group(enriched)
        display(summary)

Classification counts:


classification
adhoc_add     33
adhoc_del     33
annual_add    47
annual_del    48
Name: count, dtype: int64


Results — summary by group:


,classification,event_count,mean_excess_return,median_excess_return,win_rate,avg_holding_period_trading_days,avg_first_day_return
0,adhoc_add,16.0,0.006608,0.017015,0.625000,5.312500,-0.004561
1,adhoc_del,11.0,0.016488,-0.006874,0.454545,4.727273,-0.009020
2,annual_add,26.0,-0.006531,-0.009477,0.384615,6.000000,-0.003322
3,annual_del,25.0,0.039421,0.017626,0.760000,6.000000,0.010106


## Research

Use the cells below for ad‑hoc checks and small research (distributions, filters, plots).

In [12]:
# Example: distribution of excess return by classification
if "excess_return" in enriched.columns and "classification" in enriched.columns:
    enriched["excess_return"].describe()

In [13]:
# Example: count by classification
if "classification" in enriched.columns:
    enriched["classification"].value_counts()

In [ ]:
# Add your research cells below (plots, filters, correlations, etc.)
# e.g. enriched.hist(column="excess_return", by="classification", figsize=(8,4))